In [ ]:
# This cell is replaced automatically by scripts/run_kaggle_animator.py
# with this run's scene images + prompts before the notebook is pushed.
# Running this notebook as-is (with no SCENES injected) will just do nothing.
SCENES = {}


# WonderPop Studio — scene animator (optional, advanced)

Turns each still scene image into a short animated clip on Kaggle's free T4 GPU, using a free, openly-licensed image-to-video model via `diffusers`. Every clip is written to `/kaggle/working/` — anything in that folder is automatically downloadable afterwards via `kaggle kernels output`, which is how `scripts/run_kaggle_animator.py` retrieves it.

If this notebook ever fails to run (model name retired, out-of-memory, etc.) that's fine: the main pipeline automatically falls back to the Ken Burns image slideshow for any scene with no clip here, so a failed notebook run never blocks publishing a Short.

In [ ]:
%pip install -q diffusers transformers accelerate imageio[ffmpeg] safetensors


In [ ]:
import base64
import io
import os

import imageio
import torch
from PIL import Image
from diffusers import StableVideoDiffusionPipeline

OUT_DIR = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device != "cuda":
    print("No GPU available in this session — enable_gpu may not be set, or Kaggle's GPU quota is exhausted for this week.")

pipe = None
if device == "cuda" and SCENES:
    # Swap this for whatever free, openly-licensed img2vid checkpoint is
    # current when you're reading this — model availability shifts over
    # time faster than this guide can track.
    pipe = StableVideoDiffusionPipeline.from_pretrained(
        "stabilityai/stable-video-diffusion-img2vid",
        torch_dtype=torch.float16,
        variant="fp16",
    )
    pipe.to(device)
    pipe.enable_model_cpu_offload()


In [ ]:
NUM_FRAMES = 14      # short clip per scene — assemble_video.py loops/trims it to fit
FPS = 7
TARGET_SIZE = (576, 1024)   # keep modest for free T4 memory; upscaled later by ffmpeg

for scene_id, scene in SCENES.items():
    if pipe is None:
        break
    print(f"Animating scene {scene_id}...")
    try:
        image_bytes = base64.b64decode(scene["image_b64"])
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB").resize(TARGET_SIZE)

        result = pipe(image, num_frames=NUM_FRAMES, decode_chunk_size=4)
        frames = result.frames[0]

        out_path = os.path.join(OUT_DIR, f"scene_{scene_id}.mp4")
        imageio.mimsave(out_path, frames, fps=FPS)
        print(f"  saved {out_path}")
    except Exception as e:
        print(f"  scene {scene_id} failed: {e} — it will fall back to the static image.")

print("Done. Any scene_<id>.mp4 files above will be picked up automatically.")
